# 04 — Boruta Validation

Validates that Boruta reliably identifies truly informative features by testing it on:
1. Synthetic data with known ground truth
2. Iris (classic benchmark)
3. External solver dataset (`SampleData_0.10_threshold.csv`)

In [1]:
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.datasets import make_classification, load_iris
from boruta import BorutaPy

def run_boruta(X: pd.DataFrame, y: pd.Series, max_iter=100, perc=100,
               random_state=0) -> pd.DataFrame:
    """Run Boruta and return a ranked feature DataFrame."""
    X_ = X.replace([np.inf, -np.inf], np.nan).dropna(axis=0)
    y_ = y.loc[X_.index].astype(int).values
    rf = RandomForestClassifier(
        n_estimators=100, class_weight='balanced',
        random_state=random_state, n_jobs=-1)
    boruta = BorutaPy(
        estimator=rf, n_estimators='auto', perc=perc,
        alpha=0.05, max_iter=max_iter, random_state=random_state, verbose=0)
    boruta.fit(X_.values, y_)
    return pd.DataFrame({
        'feature':   X_.columns,
        'selected':  boruta.support_,
        'tentative': boruta.support_weak_,
        'rank':      boruta.ranking_
    }).sort_values(['rank','feature'])

def cv_score(X, y, n_splits=5):
    rf = RandomForestClassifier(n_estimators=300, random_state=0, n_jobs=-1)
    cv = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=0)
    return cross_val_score(rf, X, y, cv=cv).mean()

## 1 · Synthetic data — ground-truth recovery

In [2]:
X_syn, y_syn = make_classification(
    n_samples=5000, n_features=50, n_informative=8, n_redundant=10,
    n_clusters_per_class=2, flip_y=0.01, random_state=0)
X_syn = pd.DataFrame(X_syn, columns=[f'x{i}' for i in range(50)])
y_syn = pd.Series(y_syn)

ranking_syn = run_boruta(X_syn, y_syn, max_iter=100)
selected_syn = set(ranking_syn[ranking_syn['selected']]['feature'])

# Ground truth: x0..x7 are informative, x0..x17 include redundant (signal)
true_informative = {f'x{i}' for i in range(8)}
signal = {f'x{i}' for i in range(18)}
noise  = {f'x{i}' for i in range(18, 50)}

precision = len(selected_syn & true_informative) / max(len(selected_syn), 1)
recall    = len(selected_syn & true_informative) / len(true_informative)
noise_fpr = len(selected_syn & noise) / len(noise)

print(f'Selected: {len(selected_syn)} features')
print(f'Precision vs ground truth: {precision:.3f}')
print(f'Recall vs ground truth:    {recall:.3f}')
print(f'Noise false-positive rate: {noise_fpr:.3f}')

# CV accuracy: all features vs selected
acc_all = cv_score(X_syn, y_syn)
acc_sel = cv_score(X_syn[list(selected_syn)], y_syn) if selected_syn else 0
print(f'\nCV accuracy — all features: {acc_all:.3f}')
print(f'CV accuracy — selected:     {acc_sel:.3f}')

Selected: 18 features
Precision vs ground truth: 0.222
Recall vs ground truth:    0.500
Noise false-positive rate: 0.219

CV accuracy — all features: 0.952
CV accuracy — selected:     0.960


## 2 · Iris benchmark

In [3]:
iris = load_iris(as_frame=True)
ranking_iris = run_boruta(iris.data, iris.target, max_iter=60)
selected_iris = ranking_iris[ranking_iris['selected']]['feature'].tolist()
print('Selected Iris features:', selected_iris)
print(ranking_iris)

acc_all = cv_score(iris.data, iris.target)
acc_sel = cv_score(iris.data[selected_iris], iris.target) if selected_iris else 0
print(f'\nCV accuracy — all: {acc_all:.3f} | selected: {acc_sel:.3f}')

Selected Iris features: ['petal length (cm)', 'petal width (cm)', 'sepal length (cm)', 'sepal width (cm)']
             feature  selected  tentative  rank
2  petal length (cm)      True      False     1
3   petal width (cm)      True      False     1
0  sepal length (cm)      True      False     1
1   sepal width (cm)      True      False     1

CV accuracy — all: 0.940 | selected: 0.933


## 3 · External solver dataset

In [4]:
perf = pd.read_csv('../data/processed/Testing_with_other_dataset/SampleData_0.10_threshold.csv')
prop = pd.read_csv('../data/processed/Testing_with_other_dataset/properties_all_new.csv')

df_ext = perf.merge(prop, left_on='matrix_name', right_on='Name', how='inner')
print('External dataset shape:', df_ext.shape)

y_ext = df_ext['label'].astype(int)
X_ext = df_ext.drop(columns=['label','matrix_name','solver','Name'], errors='ignore')
X_ext = X_ext.select_dtypes(include=[np.number])
print('Features:', X_ext.shape[1], '| Class balance:', y_ext.value_counts().to_dict())

ranking_ext = run_boruta(X_ext, y_ext, max_iter=80)
selected_ext = ranking_ext[ranking_ext['selected']]['feature'].tolist()
print(f'\nSelected ({len(selected_ext)}):', selected_ext)
display(ranking_ext.head(20))

# CV comparison
acc_all = cv_score(X_ext, y_ext)
acc_sel = cv_score(X_ext[selected_ext], y_ext) if selected_ext else 0
print(f'\nCV accuracy — all: {acc_all:.3f} | selected: {acc_sel:.3f}')

External dataset shape: (62721, 42)
Features: 38 | Class balance: {0: 61573, 1: 1148}

Selected (8): ['AbsTrace', 'DiagNNZ', 'GerschgorinMin', 'LowerBW', 'RowLogValSpread', 'UpperBW', 'best_time', 'solver_time']


,feature,selected,tentative,rank
17,AbsTrace,True,False,1
30,DiagNNZ,True,False,1
37,GerschgorinMin,True,False,1
31,LowerBW,True,False,1
33,RowLogValSpread,True,False,1
32,UpperBW,True,False,1
1,best_time,True,False,1
0,solver_time,True,False,1
21,DummyRowsKind,False,False,2
28,DiagAverage,False,False,3



CV accuracy — all: 0.991 | selected: 0.993
